# Coffea Processors

Coffea relies mainly on [uproot](https://github.com/scikit-hep/uproot) to provide access to ROOT files for analysis.
As a usual analysis will involve processing tens to thousands of files, totalling gigabytes to terabytes of data, there is a certain amount of work to be done to build a parallelized framework to process the data in a reasonable amount of time. Of course, one can work directly within uproot to achieve this, as we'll show in the beginning, but coffea provides the `coffea.processor` module, which allows users to worry just about the actual analysis code and not about how to implement efficient parallelization, assuming that the parallization is a trivial map-reduce operation (e.g. filling histograms and adding them together). The module provides the following key features:

 * A `ProcessorABC` abstract base class that can be derived from to implement the analysis code;
 * A [NanoEvents](https://coffea-hep.readthedocs.io/en/latest/notebooks/nanoevents.html) interface to the arrays being read from the TTree as inputs;
 * A generic `accumulate()` utility to reduce the outputs to a single result, as showin in the accumulators notebook tutorial; and
 * A set of parallel executors to access multicore processing or distributed computing systems such as [Dask](https://distributed.dask.org/en/latest/), [Parsl](http://parsl-project.org/), [Spark](https://spark.apache.org/), [WorkQueue](https://cctools.readthedocs.io/en/latest/work_queue/), and others.

Let's start by writing a simple processor class that reads some CMS open data and plots a dimuon mass spectrum.
We'll start by copying the [ProcessorABC](https://coffea-hep.readthedocs.io/en/latest/api/coffea.processor.ProcessorABC.html#coffea.processor.ProcessorABC) skeleton and filling in some details:

 * Remove `flag`, as we won't use it
 * Adding a new histogram for $m_{\mu \mu}$
 * Building a [Candidate](https://coffea-hep.readthedocs.io/en/latest/api/coffea.nanoevents.methods.candidate.PtEtaPhiMCandidate.html#coffea.nanoevents.methods.candidate.PtEtaPhiMCandidate) record for muons, since we will read it with `BaseSchema` interpretation (the files used here could be read with `NanoAODSchema` but we want to show how to build vector objects from other TTree formats) 
 * Calculating the dimuon invariant mass

In [ ]:
import gzip
import json
from pathlib import Path

import awkward as ak
from coffea import processor
from coffea.nanoevents.methods import candidate
from hist import Hist

In [ ]:
class MyProcessor(processor.ProcessorABC):
    def __init__(self):
        pass

    def process(self, events):
        dataset = events.metadata["dataset"]
        muons = ak.zip(
            {
                "pt": events.Muon_pt,
                "eta": events.Muon_eta,
                "phi": events.Muon_phi,
                "mass": events.Muon_mass,
                "charge": events.Muon_charge,
            },
            with_name="PtEtaPhiMCandidate",
            behavior=candidate.behavior,
        )

        h_mass = (
            Hist.new.StrCat(["opposite", "same"], name="sign")
            .Log(1000, 0.2, 200.0, name="mass", label=r"$m_{\mu\mu}$ [GeV]")
            .Int64()
        )

        cut = (ak.num(muons) == 2) & (ak.sum(muons.charge, axis=1) == 0)
        # add first and second muon in every event together
        dimuon = muons[cut][:, 0] + muons[cut][:, 1]
        h_mass.fill(sign="opposite", mass=dimuon.mass)

        cut = (ak.num(muons) == 2) & (ak.sum(muons.charge, axis=1) != 0)
        dimuon = muons[cut][:, 0] + muons[cut][:, 1]
        h_mass.fill(sign="same", mass=dimuon.mass)

        return {
            dataset: {
                "entries": len(events),
                "mass": h_mass,
            }
        }

    def postprocess(self, accumulator):
        pass

If we were to just use bare uproot to execute this processor, we could do that with the following example, which:

 * Opens a CMS open data file
 * Creates a NanoEvents object using `BaseSchema` (roughly equivalent to the output of `uproot.open(...).arrays()`)
 * Creates a `MyProcessor` instance
 * Runs the `process()` function, which returns our accumulators


In [ ]:
mumu_data_filename = "root://xcache.af.uchicago.edu:1094//root://eospublic.cern.ch//eos/root-eos/cms_opendata_2012_nanoaod/Run2012B_DoubleMuParked.root"
# mumu_data_filename = "/Users/iason/fun/2021-07-06-pyhep-uproot-awkward-tutorial/data/Run2012B_DoubleMuParked.root"

In [ ]:
from coffea.nanoevents import NanoEventsFactory, BaseSchema

# mode="virtual" (the coffea 2026.7 default) reads columns lazily into
# virtual arrays and materializes them on demand -- no dask, no .compute().
events = NanoEventsFactory.from_root(
    {mumu_data_filename: "Events"},
    entry_stop=10000,
    metadata={"dataset": "DoubleMuon"},
    schemaclass=BaseSchema,
    mode="virtual",
).events()
p = MyProcessor()

In [ ]:
%%time

out = p.process(events)

In [ ]:
plot_dir = Path().cwd() / "plots"
plot_dir.mkdir(exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import mplhep

mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()
out["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")

fig.savefig(plot_dir / "dimuon_charge.png")

# Filesets
We'll need to construct a fileset to run over

In [ ]:
mumu_simulation_filename = "root://xcache.af.uchicago.edu:1094//root://eospublic.cern.ch//eos/root-eos/cms_opendata_2012_nanoaod/ZZTo4mu.root"
# mumu_simulation_filename = "/Users/iason/fun/2021-07-06-pyhep-uproot-awkward-tutorial/data/HiggsZZ4mu.root"

In [ ]:
initial_fileset = {
    "DoubleMuon": {
        "files": {
            mumu_data_filename: "Events",
        },
        "metadata": {
            "is_mc": False,
        },
    },
    "ZZ to 4mu": {
        "files": {
            mumu_simulation_filename: "Events",
        },
        "metadata": {
            "is_mc": True,
        },
    },
}

# Processing

In [ ]:
%%time

iterative_run = processor.Runner(
    executor = processor.IterativeExecutor(compression=None),
    schema=BaseSchema,
    maxchunks=3,
    savemetrics=True,
)

out, metrics = iterative_run(
    initial_fileset,
    processor_instance=MyProcessor(),
)

In [ ]:
out, metrics

In [ ]:
plot_dir = Path().cwd() / "plots"
plot_dir.mkdir(exist_ok=True)

In [ ]:
mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()
out["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")

fig.savefig(plot_dir / "dimuon_charge.png")

Now, if we want to use more than a single core on our machine, we simply change `IterativeExecutor` for `FuturesExecutor`, which uses the python `concurrent.futures` standard library. We can then set the most interesting argument to the `FuturesExecutor`: the number of cores to use

In [ ]:
%%time

futures_run = processor.Runner(
    executor = processor.FuturesExecutor(workers=4, compression=None),
    schema=BaseSchema,
    savemetrics=True,
)

out, metrics = futures_run(
    initial_fileset,
    processor_instance=MyProcessor()
)

In [ ]:
out, metrics

In [ ]:
plot_dir = Path().cwd() / "plots"
plot_dir.mkdir(exist_ok=True)

In [ ]:
mplhep.style.use(mplhep.style.ATLAS)

fig, ax = plt.subplots()
out["DoubleMuon"]["mass"].plot1d(ax=ax)
ax.set_xscale("log")
ax.legend(title="Dimuon charge")

fig.savefig(plot_dir / "dimuon_charge.png")

## Preview: upcoming coffea features

You have now run a processor through the `Runner` / executor API. The previews below show where coffea's preprocessing and execution APIs are heading.

The cells below preview features that are **not yet released** — they live in open *draft* pull requests against the coffea repository. Each demo is **guarded**: it detects whether the feature is present (by import / signature introspection, never by version number, since these are unreleased branches) and prints a gentle note instead of raising if it is missing. This section is therefore safe to "Run All" in the default environment.

To actually exercise the demos, launch one of the preview environments defined in `pixi.toml`:

```bash
pixi run -e preview jupyter lab           # pydantic dataset-tools extensions: PRs #1579, #1600, #1601
pixi run -e preview-compute jupyter lab    # the coffea.compute execution refactor: PR #1470
```

Those environments install coffea straight from the PR branches, so the exact API may drift before release.

In [ ]:
# --- Feature detection for the preview demos below ---------------------------
# These upcoming features live in *unreleased* draft PRs, so we never rely on a
# version number; each is detected by import / signature introspection. When a
# feature is absent the demo cells print a gentle note instead of raising, so
# this whole section is safe to "Run All" in the default environment.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601
HAS_COMPUTE = importlib.util.find_spec("coffea.compute") is not None  # draft PR #1470


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not available in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. To try it, launch a preview environment:\n"
        f"            pixi run -e preview jupyter lab           # pydantic dataset-tools extensions (#1579/#1600/#1601)\n"
        f"            pixi run -e preview-compute jupyter lab    # coffea.compute execution refactor (#1470)"
    )


# A small, network-free sample so the preview demos run wherever this repo is
# checked out (they fall back gracefully if it is missing).
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
_preview_fileset = {
    "demo": {"files": {str(_preview_file): "Events"}, "metadata": {"xsec": 1.0}}
}

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS", "HAS_COMPUTE"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

Filesets can now be expressed as validated `pydantic` models (`DataGroupSpec` / `DatasetSpec` / `ROOTFileSpec`, ...). Malformed filesets fail fast with clear errors, and the models carry form and metadata around for the tools below.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# The classic "dict-in / dict-out" fileset still works, but datasets can now be
# expressed as *validated* pydantic models, catching malformed filesets early.
from coffea.dataset_tools import ModelFactory, DatasetSpec

spec = ModelFactory.dict_to_datasetspec(_preview_fileset["demo"])
print("type:", type(spec).__name__, "| is DatasetSpec:", isinstance(spec, DatasetSpec))
print("validated metadata:", dict(spec.metadata))
print("file specs:", [type(fs).__name__ for fs in spec.files.values()])
# round-trip back to a plain dict when a legacy API needs one
_roundtrip = ModelFactory.datasetspec_to_dict(spec)

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    available, report = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    finfo = list(available["demo"]["files"].values())[0]
    print("preprocessed with the dask-free 'iterative' backend")
    print("  steps discovered:", finfo["steps"])
    print("  num_entries:", finfo["num_entries"])
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into dataset-level metadata
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    print("dataset metadata after extraction:", dict(available["demo"]["metadata"]))
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import resizable_steps

    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            # after the first chunk, ask the generator to shrink the step to 100
            produced.append(gen.send(100))
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # Higher-level driver, operating on a preprocessed pydantic DatasetSpec:
    print(
        "\nHigher-level API (illustrative):\n"
        "    from coffea.dataset_tools.mutable_steps import (\n"
        "        iter_dataset_steps, run_adaptive_steps, WallTimeStepPolicy)\n"
        "    policy = WallTimeStepPolicy(target_seconds=30)\n"
        "    total = run_adaptive_steps(dataset_spec, work_fn, step_size=100_000, policy=policy)"
    )
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")

### 5. A unified execution protocol: `coffea.compute` — draft PR #1470

The largest change on the horizon. PR #1470 introduces `coffea.compute`, replacing the `Processor` / `Executor` / `Runner` trio with a single `Backend` **protocol**. Work is expressed as a `Computable` (a `Dataset` mapped through a function via `.map_steps`), handed to any backend's `.compute()`, which returns a non-blocking `Task` exposing `.result()`, `.partial_result()`, `.wait()`, and `.cancel()`. The preprocessing backends (#1579) and resizable steps (#1601) are stepping stones toward this unified interface. The one-liner it enables:

```python
with ThreadedBackend() as backend:
    total = backend.compute(dataset.map_steps(process)).result()
```

This backend is a genuine work in progress, so the demo below is guarded to show the protocol *shape* even where it does not yet fully execute.

In [ ]:
# 5. A unified execution protocol: coffea.compute  (draft PR #1470, WIP)
# PR #1470 replaces the Processor/Executor/Runner trio with a single `Backend`
# protocol. A `Computable` (a Dataset mapped through a function via .map_steps)
# is handed to any backend's .compute(), returning a non-blocking Task with
# .result()/.partial_result(). This is what tutorial scaleout could look like
# once the refactor lands:
if HAS_COMPUTE:
    from coffea.compute.data import Dataset, File, ContextDataset
    from coffea.compute.backends.threaded import ThreadedBackend

    dataset = Dataset(
        files=[File(path=str(_preview_file), steps=[(0, 50_000), (50_000, 100_000)])],
        metadata=ContextDataset(dataset_name="demo", cross_section=None),
    )

    def count(events):  # a plain callable *is* the processor
        return len(events)

    computable = dataset.map_steps(count)
    print(f"built a Computable with {len(computable)} work element(s)")
    print(
        "the target one-liner:\n"
        "    with ThreadedBackend() as backend:\n"
        "        total = backend.compute(computable).result()"
    )
    try:
        with ThreadedBackend() as backend:
            total = backend.compute(computable).result()
        print("result:", total)
    except Exception as exc:  # coffea.compute is a work-in-progress preview
        print(
            f"[preview] coffea.compute did not execute here yet ({type(exc).__name__}); "
            "the protocol shape above is the point of this WIP preview."
        )
else:
    preview_note("coffea.compute", "#1470")